## Assignment 4

*100 points (8% of course grade)*</br>
*Assigned: Mon, Nov 16th*</br>
**Due: Wed, Nov 25th, 23:59**

This homework covers mongodb queries, and you may need to spend some time in setting up your MongoDB (will come back to Postgres in A5!). Some programming details are not covered in the lectures, so you will need to read the documentations and tutorials yourself. If you wait until the last minute, you might be overwhelmed.

You must turn in the required files electronically. Please follow the submission instructions at the end of this notebook.

## Setup the `parliament` Database

Consider information about the Parliament of Canada stored as JSON documents in a MongoDB database named `parliament` (House of Commons and Senate, data as of November, 2026). To start the MongoDB database server and construct this database, follow the instructions in the [MongoDB setup instruction](https://canvas.sfu.ca/courses/16213/pages/6-mongodb), and then use the following commands:

```bash
(enter your A4/parliament folder; you don't need this if you are using the Docker container)
mongorestore --gzip --archive=parliament/mongodb-dump.tgz 
```
The database contains two types of documents inside two collections, people and committees. To see these documents, use the following commands (warning: the entire database may be too long to print on your terminal):
```bash
mongosh parliament --quiet --eval 'printjson(db.people.find().toArray())'
mongosh parliament --quiet --eval 'printjson(db.committees.find().toArray())'
```
If you are using our Docker container, use the full connection string URI for the db address (you will need this for all mongo commands!)
```bash
mongorestore mongodb://mongodb:27017/ --drop --gzip --archive=parliament/mongodb-dump.tgz # You don't need this if you followed the instructions for the Docker Container; just in case if you want to restore the database.
mongosh mongodb://mongodb:27017/parliament --quiet --eval 'printjson(db.people.find().toArray())'
mongosh mongodb://mongodb:27017/parliament --quiet --eval 'printjson(db.people.find().toArray())'
```


The structures of these documents are self-explanatory. The database has two collections.

- Each document in `people` collection stores information about a legislator, a Member of Parliament (MP, House of Commons) or a senator. Its `_id` is a string such as `ca:person:commons:96081` or `ca:person:senate:2811`.
  - `roles[]` lists legislative service. `type` is `"mp"` or `"senator"`. A role is current if its `temporal_status` is `"current"`. `start` and `end` are **strings** in `"YYYY-MM-DD"` form, and ongoing service has `end: null`. MP roles have `parliament`, `province` and `constituency.name` (the riding). Senator roles have `province`, `province_code` and `expected_retirement_date`.
  - `affiliations[]` holds party (House) or parliamentary group (Senate) memberships, each with `chamber`, `name`, `kind`, `start`, `end` and `temporal_status`.
- Each document in `committees` collection stores information about a committee. It has `code` (`FAAE`), `chamber` (`commons`, `senate` or `joint`), `name`, etc.
  - It has a list of members, whose ids reference those of people. `role` specifies the role of the member in the committee (`chair`, `vice_chair`, `deputy_chair`, `joint_chair`, `member`, `associate_member`). `members[]` stores the **dated history**: appointments that ended during the session stay in the list with `temporal_status: "ended"`.
  - Oftentimes a committee can have subcommittees. Each subcommittee element has its own list of members. A legislator can serve on multiple committees, and even multiple subcommittees under the same committee. 

Note that the documents are identified by their `_id` attribute values.


## Problem 1: Querying MongoDB (70%)

Write MongoDB queries (in MongoDB shell syntax) to answer the following questions. Unless otherwise noted, please make sure that your answer appears as an array. Your query should have the form `printjson(db.collection.method(...).toArray())`, where collection is one of `people` and `committees`, and method is one of `find` and `aggregate`. For each question below, say (a), write your MongoDB query in the corresponding files (a.js, b.js, ... ). You can run your query directly in mongodb shell
```bash
mongosh parliament --quiet -f a.js
```
or, in the Docker container,
```bash
mongosh mongodb://mongodb:27017/parliament --quiet -f a.js
```

#### a. (8 points) Find legislators with the exact last name of "Martin". Simply print the entire person documents.
You can use `attr:/pattern/` to match the value of attr against a regular expression pattern. In particular, `/ XYZ$/` (note the space before XYZ) ensures that the string ends with a space followed by “XYZ”; `"$"` in the pattern matches the end of the string.


#### b. (8 points) List all current legislators born after the first version of Microsoft SQL Server released (April 24, 1989) but before the first version of Postgres released (July 8, 1996). Format each of them simply as `{"name": "...", "birth_date": "..."}`. Order them by the name attribute.

Dates of birth are stored in `birth_date` as `"YYYY-MM-DD"` strings, which sort in date order, so compare them with strings, e.g., use `attr: { $gt: "2000-01-01" }` to test if the value of attr is later than the date 2000-01-01.

#### c. (8 points) List the names of current Liberal MPs who at some point earlier sat in the House of Commons under a different affiliation. Format each of them as `{ "name": "…" }`. Order them by name (ascending).


#### d. (10 points) List all current female Conservative legislators. Format each of them as `{ "name": "…", "age": …, "province": "…", "type": "mp_or_senator" }`, where age is an approximation obtained by subtracting their birth year (`birth_year`) from 2026. Order them by age (descending), with ties broken by name (ascending). Skip those whose year of birth is not recorded.

Use the current affiliation, since some MPs changed party during the session. `type` is the type of the current role. You can use `$subtract` to perform subtraction. 


#### e. (12 points) Find the deputy chair (role of "deputy_chair") of the Senate "Subcommittee on Human Resources" (under committee CIBA, the "Standing Committee on Internal Economy, Budgets and Administration"). Simply print the entire person document.


#### f. (12 points) List the names of legislators (MPs and senators) who are currently serving British Columbia but do NOT hold a current seat on any committee (ignore subcommittees). Any role counts as a seat, associate membership included; appointments with `temporal_status: "ended"` do not count. Format each of them as `{ "name": "…" }`. Order them by name (ascending).

#### g. (12 points) Find the number of current legislators (MPs and senators) for each province and territory (use the `province` of the current role) by gender. Sort the provinces and territories by name alphabetically. Your output should look like the following (whitespace is unimportant):
```
[
  { province: 'Alberta', M: 33, F: 10 },
  { province: 'British Columbia', M: 36, F: 11 },
…]
```


## Problem 2.  Application programming with MongoDB (30%)

MongoDB is also a popular option for supporting web applications. In this part, your job is to implement a Web API like you did in A3. But this time we will use MongoDB instead of Postgres. Although MongoDB is for JSON documents, it is still a popular choice to use Flask as the backend framework, and we also want to keep your learning curve moderate.


Please first refer to the basic knowledge about Flask in A3. Here we just list the setup.

**Step 0.** Set up your Python environment (installation instructions were in Assignment 1's setup).
  * In your terminal, run:
    ```bash
    pip install Flask Flask-PyMongo
    ```
    for the most recent versions.
  * You may specify the version of each library with `library==version`, e.g. `Flask-PyMongo==3.0.1`. The reference solution was tested with Flask 3.1.3, Flask-PyMongo 3.0.1 and PyMongo 4.20.
  * If you are using the Docker container, we already provided a conda environment: `conda activate mongo`.
  * `app.py` connects to `mongodb://localhost:27017/parliament`. In the Docker container, change `MONGO_URI` to `mongodb://mongodb:27017/parliament`.
  * Start the server with `python app.py`; it listens on http://127.0.0.1:7000.
  * You may either open the testing url in a browser, or run `curl [the url]` in your terminal.


### Tasks: finish a and b below.

You may create other functions you need and don't have to put everything in the required functions. But **DON'T add more files or refer to other Python libraries**. We still work on the `parliament` database as in P1. One example method is provided in `app.py`:

* http://127.0.0.1:7000/person/ca:person:commons:96081
- Response: ```{"_id": "ca:person:commons:96081", "chamber": "commons", "name": "Anita Anand"}```

#### a. (14 points) Provide a method to get the name, riding, and party of each current MP of the given province or territory (`province_code`, a two-letter code such as `BC`). Format each of them in the form `{ "name": "…", "riding": "…", "party": "…" }` and sort them according to the riding.

Complete the `getByProvince()` function in `app.py`. Follow the instructions in the comments.

* MP roles store the province name not the code； `app.py` provides a `PROVINCES` dictionary from codes to names. Return a 404 error for an unknown code.
* `party` is the `name` of the MP's **current** House affiliation (`chamber: "commons"`, `temporal_status: "current"`). 

**Test**
* http://127.0.0.1:7000/getByProvince/NU
- Response: ```[{"name": "Lori Idlout", "riding": "Nunavut", "party": "Liberal"}]```
* http://127.0.0.1:7000/getByProvince/BC
- Response: 43 MPs, starting ```[{"name": "Sukhman Gill", "riding": "Abbotsford—South Langley", "party": "Conservative"}, {"name": "Wade Chang", "riding": "Burnaby Central", "party": "Liberal"}, ...]```
* http://127.0.0.1:7000/getByProvince/XX
- Response: 404 Error

#### b. (16 points) Provide a method to update the information of a given legislator (`lid`) as of a given date (`as_of`, a `"YYYY-MM-DD"` string): `/update/<lid>/<as_of>`.

A current role (`temporal_status: "current"`) is over if its `end` date or, for a senator, its `expected_retirement_date` (the senator's 75th birthday, when they must retire) is before `as_of`. For such a role, set `temporal_status` to `"ended"` and `end` to that date. Dates are `"YYYY-MM-DD"` strings, so compare them as strings. Ongoing MP roles have `end: null` and stay current. You may assume that `as_of` is a valid date.

If the lid does not exist, return a 404 error; return the updated legislator object if the record is updated, and update the database accordingly; return `{'res': 'current'}` if there is no need to update. You don't need to worry about inserting new roles or changing `affiliations[]`. Also, there is no need to remove anyone from the committees collection.

The statements that you need to complete are marked with comments in uppercase (three pieces in total).

Note: this operation may change some of the query results in P1, so you may want to finish P1 and then work on this part, or remember to reload the database after testing this part.

Complete the `update()` function in `app.py`. Hint: you may use the functions `find_one()` and `update_one()` (see [the doc](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/#std-label-pymongo-crud)).

**Test**

* http://127.0.0.1:7000/update/ca:person:senate:206019/2027-07-07 no update (`{"res": "current"}`): Peter Harder retires on 2027-08-25
* http://127.0.0.1:7000/update/ca:person:senate:206019/2027-11-16 updated: the senator role now has `"end": "2027-08-25"` and `"temporal_status": "ended"`
* http://127.0.0.1:7000/update/ca:person:commons:96081/2030-01-01 no update (an MP: `end` is null, so `{"res": "current"}`)
* http://127.0.0.1:7000/update/ca:person:nobody/2027-07-07 404 Not Found


## Submission instruction

1. For problem 1, fill your solutions in `a.js` through `g.js` (all in folder `p1`)

2. For problem 2, fill your solutions in `app.py` (in folder `p2`)

3. Compress folders `p1` and `p2` into `A4.zip` and submit on Canvas (No solution needed in the Jupyter Notebook!)

4. Remember to download your submissions to double check that your files are up-to-date! DO NOT submit the original empty files! 